# 01 — IoT Data Simulation & Generation
**AI-Driven Waste Collection & Route Optimization**  
Birla Institute of Technology - Mesra  

This notebook simulates IoT sensor data from smart bins across a city grid.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
import random
import json
import os

np.random.seed(42)
random.seed(42)
print('Libraries loaded successfully.')

## 1. City Grid & Bin Configuration

In [ ]:
# City configuration
CITY_CONFIG = {
    'name': 'SmartCity_Demo',
    'num_bins': 50,
    'num_trucks': 5,
    'grid_size_km': 10,
    'depot_location': (5.0, 5.0),
    'sensor_interval_min': 15,
    'bin_capacity_liters': 240,
    'high_priority_threshold': 0.80,
}

ZONE_TYPES = {
    'residential': {'fill_rate_mean': 2.5, 'fill_rate_std': 0.8, 'count': 25},
    'commercial':  {'fill_rate_mean': 5.2, 'fill_rate_std': 1.5, 'count': 15},
    'industrial':  {'fill_rate_mean': 8.1, 'fill_rate_std': 2.0, 'count': 7},
    'park':        {'fill_rate_mean': 1.2, 'fill_rate_std': 0.4, 'count': 3},
}

print(f"City: {CITY_CONFIG['name']}")
print(f"Total bins: {CITY_CONFIG['num_bins']}")
print(f"Trucks: {CITY_CONFIG['num_trucks']}")
print(f"Grid: {CITY_CONFIG['grid_size_km']} x {CITY_CONFIG['grid_size_km']} km")

## 2. Generate Bin Locations & Metadata

In [ ]:
bins = []
bin_id = 0
for zone, cfg in ZONE_TYPES.items():
    for i in range(cfg['count']):
        x = np.random.uniform(0.5, 9.5)
        y = np.random.uniform(0.5, 9.5)
        bins.append({
            'bin_id': f'BIN_{bin_id:03d}',
            'zone_type': zone,
            'x_coord': round(x, 3),
            'y_coord': round(y, 3),
            'capacity_liters': CITY_CONFIG['bin_capacity_liters'],
            'fill_rate_pct_per_hr': cfg['fill_rate_mean'] + np.random.normal(0, cfg['fill_rate_std']),
            'sensor_type': 'ultrasonic',
            'install_date': '2024-01-01',
        })
        bin_id += 1

bins_df = pd.DataFrame(bins)
os.makedirs('../data', exist_ok=True)
bins_df.to_csv('../data/bins_metadata.csv', index=False)

print(bins_df.groupby('zone_type')[['bin_id','fill_rate_pct_per_hr']].agg({'bin_id':'count','fill_rate_pct_per_hr':'mean'}).rename(columns={'bin_id':'count'}))
bins_df.head()

## 3. Simulate Time-Series Fill Levels (30 days)

In [ ]:
def simulate_fill_level(bin_row, start_dt, num_days=30, interval_min=15):
    records = []
    current_dt = start_dt
    fill_pct = np.random.uniform(5, 30)
    steps = (num_days * 24 * 60) // interval_min

    for _ in range(steps):
        hr = current_dt.hour
        dow = current_dt.weekday()

        # Time-of-day and day-of-week multipliers
        tod_mult = 1.5 if 7 <= hr <= 10 else (1.8 if 12 <= hr <= 14 else (1.3 if 17 <= hr <= 20 else 0.3))
        dow_mult = 1.4 if dow in [5, 6] else 1.0

        base_rate = bin_row['fill_rate_pct_per_hr'] * (interval_min / 60)
        increment = base_rate * tod_mult * dow_mult + np.random.normal(0, 0.3)
        fill_pct = min(100, max(0, fill_pct + max(0, increment)))

        records.append({
            'timestamp': current_dt,
            'bin_id': bin_row['bin_id'],
            'fill_level_pct': round(fill_pct, 2),
            'zone_type': bin_row['zone_type'],
        })

        if fill_pct >= 95:
            fill_pct = np.random.uniform(3, 10)  # simulate emptying

        current_dt += timedelta(minutes=interval_min)

    return records

start = datetime(2024, 3, 1)
all_records = []
for _, row in bins_df.iterrows():
    all_records.extend(simulate_fill_level(row, start, num_days=30))

sensor_df = pd.DataFrame(all_records)
sensor_df.to_csv('../data/sensor_readings.csv', index=False)
print(f'Total sensor readings: {len(sensor_df):,}')
print(f'Date range: {sensor_df.timestamp.min()} → {sensor_df.timestamp.max()}')
sensor_df.describe()

## 4. Visualize Fill Level Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('IoT Sensor Fill Level Patterns — Smart Waste City Demo', fontsize=14, fontweight='bold')

# --- Plot 1: Sample bins time series
ax = axes[0, 0]
sample_bins = bins_df.sample(4)['bin_id'].tolist()
sample_data = sensor_df[sensor_df['bin_id'].isin(sample_bins)]
for b in sample_bins:
    sub = sample_data[sample_data['bin_id'] == b].head(192)
    ax.plot(sub['timestamp'], sub['fill_level_pct'], label=b, linewidth=1)
ax.set_title('Fill Level Time-Series (4 Sample Bins, 48h)')
ax.set_ylabel('Fill Level (%)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# --- Plot 2: Avg fill by hour-of-day
ax = axes[0, 1]
sensor_df['hour'] = pd.to_datetime(sensor_df['timestamp']).dt.hour
hourly = sensor_df.groupby('hour')['fill_level_pct'].mean()
ax.bar(hourly.index, hourly.values, color='#00BFA5', alpha=0.85)
ax.set_title('Avg Fill Level by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Avg Fill (%)')
ax.grid(True, alpha=0.3, axis='y')

# --- Plot 3: Fill distribution by zone
ax = axes[1, 0]
for zone in sensor_df['zone_type'].unique():
    data = sensor_df[sensor_df['zone_type'] == zone]['fill_level_pct']
    ax.hist(data, bins=40, alpha=0.6, label=zone, density=True)
ax.set_title('Fill Level Distribution by Zone Type')
ax.set_xlabel('Fill Level (%)')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 4: City bin map colored by avg fill
ax = axes[1, 1]
avg_fill = sensor_df.groupby('bin_id')['fill_level_pct'].mean().reset_index()
merged = bins_df.merge(avg_fill, on='bin_id')
zone_colors = {'residential': 'blue', 'commercial': 'orange', 'industrial': 'red', 'park': 'green'}
for zone in merged['zone_type'].unique():
    sub = merged[merged['zone_type'] == zone]
    sc = ax.scatter(sub['x_coord'], sub['y_coord'], c=sub['fill_level_pct'],
                    cmap='RdYlGn_r', vmin=20, vmax=80, s=60, label=zone, alpha=0.85)
plt.colorbar(sc, ax=ax, label='Avg Fill %')
depot = CITY_CONFIG['depot_location']
ax.plot(*depot, 'k*', markersize=15, label='Depot')
ax.set_title('City Bin Map — Avg Fill Level')
ax.set_xlabel('X (km)')
ax.set_ylabel('Y (km)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/01_iot_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')